Prompt:

• Propose and apply a method for identifying or highlighting ambiguous or potentially misclassified samples using accessible and interpretable techniques.

In [4]:
import os
import json
import random
from dataclasses import dataclass, asdict
from typing import Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# =========================
# CONFIG
# =========================
@dataclass
class Config:
    seed: int = 42
    batch_size: int = 64
    epochs: int = 30
    learning_rate: float = 1e-3
    test_size: float = 0.2
    val_size: float = 0.2  # fraction of train_val split

    # ambiguity thresholds for binary classification
    confidence_threshold: float = 0.60
    margin_threshold: float = 0.20  # distance from 0.5 decision boundary * 2

    data_path: str = "bank-full.csv"
    target_column: str = "y"
    positive_label: str = "yes"

    output_dir: str = "outputs/part_b_bank"
    model_path: str = "outputs/part_b_bank/bank_part_b_model.keras"
    history_path: str = "outputs/part_b_bank/history.json"
    metrics_path: str = "outputs/part_b_bank/metrics.json"
    flagged_path: str = "outputs/part_b_bank/flagged_samples.csv"
    preprocessor_path: str = "outputs/part_b_bank/preprocessor_columns.json"


CFG = Config()


# =========================
# REPRODUCIBILITY
# =========================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# =========================
# IO
# =========================
def ensure_output_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# =========================
# DATA LOADING / PREP
# =========================
def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";")
    return df


def split_features_target(df: pd.DataFrame, target_column: str, positive_label: str):
    X = df.drop(columns=[target_column]).copy()
    y = (df[target_column] == positive_label).astype(int).values
    return X, y


def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
    numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ])
    return preprocessor


def prepare_data(df: pd.DataFrame):
    X, y = split_features_target(df, CFG.target_column, CFG.positive_label)

    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=CFG.test_size,
        stratify=y,
        random_state=CFG.seed,
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=CFG.val_size,
        stratify=y_train_val,
        random_state=CFG.seed,
    )

    preprocessor = build_preprocessor(X_train)
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)
    X_test_processed = preprocessor.transform(X_test)

    feature_names = get_feature_names(preprocessor)

    return (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        X_train_processed.astype(np.float32),
        X_val_processed.astype(np.float32),
        X_test_processed.astype(np.float32),
        preprocessor,
        feature_names,
    )


def get_feature_names(preprocessor: ColumnTransformer) -> List[str]:
    feature_names = []
    for name, transformer, cols in preprocessor.transformers_:
        if name == "num":
            feature_names.extend(cols)
        elif name == "cat":
            onehot = transformer.named_steps["onehot"]
            feature_names.extend(onehot.get_feature_names_out(cols).tolist())
    return feature_names


# =========================
# EDA
# =========================
def run_basic_eda(df: pd.DataFrame) -> Dict:
    summary = {
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "missing_values": df.isna().sum().to_dict(),
        "class_distribution": df[CFG.target_column].value_counts().to_dict(),
        "numeric_summary": df.describe(include=[np.number]).to_dict(),
    }
    return summary


def plot_target_distribution(df: pd.DataFrame):
    counts = df[CFG.target_column].value_counts()
    plt.figure(figsize=(6, 4))
    counts.plot(kind="bar")
    plt.title("Target Distribution")
    plt.xlabel("Subscription Outcome")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.output_dir, "target_distribution.png"), dpi=200)
    plt.close()


def plot_numeric_histograms(df: pd.DataFrame, columns: List[str]):
    for col in columns:
        plt.figure(figsize=(6, 4))
        df[col].hist(bins=30)
        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.tight_layout()
        plt.savefig(os.path.join(CFG.output_dir, f"hist_{col}.png"), dpi=200)
        plt.close()


# =========================
# MODEL
# =========================
def build_model(input_dim: int) -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.30),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.25),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(1, activation="sigmoid"),
    ])

    optimizer = keras.optimizers.Adam(learning_rate=CFG.learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )
    return model


# =========================
# TRAINING
# =========================
def train_model(model: keras.Model, X_train, y_train, X_val, y_val):
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_auc",
            mode="max",
            patience=5,
            restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
        ),
    ]

    class_counts = np.bincount(y_train)
    total = len(y_train)
    class_weight = {
        0: total / (2 * class_counts[0]),
        1: total / (2 * class_counts[1]),
    }

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=CFG.epochs,
        batch_size=CFG.batch_size,
        callbacks=callbacks,
        class_weight=class_weight,
        verbose=1,
    )
    return history


# =========================
# EVALUATION / AMBIGUITY
# =========================
def get_predictions(model: keras.Model, X_test: np.ndarray):
    probs = model.predict(X_test, verbose=0).flatten()
    preds = (probs >= 0.5).astype(int)
    confidence = np.where(probs >= 0.5, probs, 1 - probs)
    margin = np.abs(probs - 0.5) * 2
    return probs, preds, confidence, margin


def flag_ambiguous_samples(
    X_test_raw: pd.DataFrame,
    y_true: np.ndarray,
    probs: np.ndarray,
    preds: np.ndarray,
    confidence: np.ndarray,
    margin: np.ndarray,
) -> pd.DataFrame:
    flagged_mask = (confidence < CFG.confidence_threshold) | (margin < CFG.margin_threshold)

    flagged = X_test_raw.loc[flagged_mask].copy()
    flagged["true_label"] = y_true[flagged_mask]
    flagged["predicted_label"] = preds[flagged_mask]
    flagged["predicted_probability_yes"] = probs[flagged_mask]
    flagged["prediction_confidence"] = confidence[flagged_mask]
    flagged["decision_margin"] = margin[flagged_mask]
    flagged["misclassified"] = y_true[flagged_mask] != preds[flagged_mask]
    flagged["reason"] = np.where(
        confidence[flagged_mask] < CFG.confidence_threshold,
        "low_confidence",
        "close_to_decision_boundary",
    )
    return flagged


def calculate_metrics(y_true: np.ndarray, preds: np.ndarray, probs: np.ndarray, flagged_df: pd.DataFrame) -> Dict:
    cm = confusion_matrix(y_true, preds)
    total_errors = int((y_true != preds).sum())
    flagged_count = int(len(flagged_df))
    flagged_errors = int(flagged_df["misclassified"].sum()) if flagged_count > 0 else 0

    return {
        "accuracy": float((y_true == preds).mean()),
        "precision": float(precision_score(y_true, preds, zero_division=0)),
        "recall": float(recall_score(y_true, preds, zero_division=0)),
        "f1": float(f1_score(y_true, preds, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probs)),
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(y_true, preds, output_dict=True, zero_division=0),
        "total_errors": total_errors,
        "flagged_count": flagged_count,
        "flagged_rate": float(flagged_count / len(y_true)),
        "flagged_error_precision": float(flagged_errors / flagged_count) if flagged_count > 0 else 0.0,
        "flagged_error_recall": float(flagged_errors / total_errors) if total_errors > 0 else 0.0,
    }


# =========================
# PLOTS
# =========================
def plot_training_history(history):
    hist = history.history

    plt.figure(figsize=(8, 5))
    plt.plot(hist["accuracy"], label="train_accuracy")
    plt.plot(hist["val_accuracy"], label="val_accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training vs Validation Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.output_dir, "training_accuracy.png"), dpi=200)
    plt.close()

    plt.figure(figsize=(8, 5))
    plt.plot(hist["loss"], label="train_loss")
    plt.plot(hist["val_loss"], label="val_loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.output_dir, "training_loss.png"), dpi=200)
    plt.close()


def plot_confusion(cm: np.ndarray):
    plt.figure(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["no", "yes"])
    disp.plot(cmap="Blues", colorbar=False)
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.output_dir, "confusion_matrix.png"), dpi=200)
    plt.close()


def plot_confidence_histogram(confidence: np.ndarray, correctness: np.ndarray):
    plt.figure(figsize=(8, 5))
    plt.hist(confidence[correctness], bins=25, alpha=0.7, label="Correct")
    plt.hist(confidence[~correctness], bins=25, alpha=0.7, label="Incorrect")
    plt.xlabel("Prediction Confidence")
    plt.ylabel("Count")
    plt.title("Confidence Distribution: Correct vs Incorrect")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.output_dir, "confidence_histogram.png"), dpi=200)
    plt.close()


def plot_margin_histogram(margin: np.ndarray, correctness: np.ndarray):
    plt.figure(figsize=(8, 5))
    plt.hist(margin[correctness], bins=25, alpha=0.7, label="Correct")
    plt.hist(margin[~correctness], bins=25, alpha=0.7, label="Incorrect")
    plt.xlabel("Decision Margin")
    plt.ylabel("Count")
    plt.title("Decision Margin Distribution: Correct vs Incorrect")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(CFG.output_dir, "margin_histogram.png"), dpi=200)
    plt.close()


# =========================
# MAIN
# =========================
def main():
    set_seed(CFG.seed)
    ensure_output_dir(CFG.output_dir)

    with open(os.path.join(CFG.output_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(asdict(CFG), f, indent=4)

    df = load_data(CFG.data_path)
    eda_summary = run_basic_eda(df)

    with open(os.path.join(CFG.output_dir, "eda_summary.json"), "w", encoding="utf-8") as f:
        json.dump(eda_summary, f, indent=4)

    plot_target_distribution(df)
    plot_numeric_histograms(df, ["age", "balance", "duration", "campaign"])

    (
        X_train_raw,
        X_val_raw,
        X_test_raw,
        y_train,
        y_val,
        y_test,
        X_train,
        X_val,
        X_test,
        preprocessor,
        feature_names,
    ) = prepare_data(df)

    with open(CFG.preprocessor_path, "w", encoding="utf-8") as f:
        json.dump(feature_names, f, indent=4)

    model = build_model(input_dim=X_train.shape[1])
    model.summary()

    history = train_model(model, X_train, y_train, X_val, y_val)
    model.save(CFG.model_path)

    with open(CFG.history_path, "w", encoding="utf-8") as f:
        json.dump(history.history, f, indent=4)

    probs, preds, confidence, margin = get_predictions(model, X_test)
    flagged_df = flag_ambiguous_samples(X_test_raw, y_test, probs, preds, confidence, margin)
    flagged_df.to_csv(CFG.flagged_path, index=False)

    metrics = calculate_metrics(y_test, preds, probs, flagged_df)
    with open(CFG.metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=4)

    correctness = preds == y_test
    cm = np.array(metrics["confusion_matrix"])

    plot_training_history(history)
    plot_confusion(cm)
    plot_confidence_histogram(confidence, correctness)
    plot_margin_histogram(margin, correctness)

    print("\n=== PART B BANK MARKETING SUMMARY ===")
    print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    print(f"Accuracy: {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1: {metrics['f1']:.4f}")
    print(f"ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"Flagged samples: {metrics['flagged_count']} / {len(y_test)} ({metrics['flagged_rate']:.2%})")
    print(f"Flagged error precision: {metrics['flagged_error_precision']:.4f}")
    print(f"Flagged error recall: {metrics['flagged_error_recall']:.4f}")


if __name__ == "__main__":
    main()


/var/folders/87/p5k8cs9s6bz_6qk2rg6t98bc0000gn/T/ipykernel_72392/1306052526.py:92: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         6,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,793 (69.50 KB)

 Trainable params: 17,409 (68.00 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.7501 - auc: 0.8582 - loss: 0.4771 - precision: 0.2966 - recall: 0.8284 - val_accuracy: 0.8214 - val_auc: 0.9186 - val_loss: 0.3565 - val_precision: 0.3846 - val_recall: 0.8783 - learning_rate: 0.0010
Epoch 2/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7961 - auc: 0.9016 - loss: 0.3912 - precision: 0.3507 - recall: 0.8733 - val_accuracy: 0.8148 - val_auc: 0.9211 - val_loss: 0.3505 - val_precision: 0.3766 - val_recall: 0.8913 - learning_rate: 0.0010
Epoch 3/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 0s 848us/step - accuracy: 0.8019 - auc: 0.9085 - loss: 0.3760 - precision: 0.3601 - recall: 0.8922 - val_accuracy: 0.8152 - val_auc: 0.9231 - val_loss: 0.3561 - val_precision: 0.3784 - val_recall: 0.9031 - learning_rate: 0.0010
Epoch 4/30
453/453 ━━━━━━━━━━━━━━━━━━━━ 0s 844us/step - accuracy: 0.8048 - auc: 0.9146 - loss: 0.3607 - precision: 0.3640 - recall: 0.8945 - val_accuracy: 0.8134 - val_auc: 0.9253 - val_loss: 0.3577 -

<Figure size 500x400 with 0 Axes>